# Criptoanálisis diferencial cuántico experimental sobre SPN16

Este notebook sigue el modelo Q1 del capítulo 6. Los pares de textos y sus cifrados se obtienen clásicamente. Los circuitos cuánticos procesan la tabla de pares ya construida y el espacio explícito $\mathcal K_r$.

La demostración implementa $\mathrm{RP}(y_r,i)$, $R(y_r)$, el oráculo $O_2$, el conteo $\widetilde R(y_r)$, el oráculo umbral $O_1$ y una búsqueda de máximo tipo Dürr--Høyer.

In [87]:
from pathlib import Path
import importlib
import sys

current = Path.cwd().resolve()
ROOT = None
for path in [current, *current.parents]:
    if (path / "quantum_differential_cryptanalysis").is_dir():
        ROOT = path
        break
    if path.name == "quantum_differential_cryptanalysis":
        ROOT = path.parent
        break
if ROOT is None:
    raise RuntimeError("No se encontró la raíz del repositorio")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from quantum_differential_cryptanalysis.src.classical_attack import (
    count_right_pairs, encrypt_pairs, exhaustive_key_search,
    generate_plaintext_pairs, right_pair,
)
from quantum_differential_cryptanalysis.src.durr_hoyer_max import durr_hoyer_max
from quantum_differential_cryptanalysis.src.oracles import build_O1, build_O2
from quantum_differential_cryptanalysis.src.quantum_counting import quantum_counting
from quantum_differential_cryptanalysis.src.spn16 import (
    SBOX, INV_SBOX, decrypt_last_round_partial, encrypt_block,
    encryption_trace, split_master_key,
)
# Recargar utils evita usar una versión antigua conservada por el kernel.
importlib.invalidate_caches()
import quantum_differential_cryptanalysis.src.utils as qdiff_utils
qdiff_utils = importlib.reload(qdiff_utils)

embed_active_candidate = qdiff_utils.embed_active_candidate
format_hex16 = qdiff_utils.format_hex16
format_partial_key = qdiff_utils.format_partial_key
project_active_candidate = qdiff_utils.project_active_candidate

print(f"Repo root: {ROOT}")

Repo root: C:\Users\juanc\github\Differential-Cryptanalysis-


## 1. Definición del SPN16

Se usan cuatro rondas y cinco subclaves de 16 bits. Las tres primeras rondas aplican `XOR -> SBOX -> PBOX`; la última aplica `XOR -> SBOX -> XOR K5`. Por tanto, la candidata $y_r$ del ataque es la subclave final `K5`.

In [88]:
ROUNDS = 4
MASTER_KEY_HEX = "00112233445566778899"
alpha = 0x0B00
alpha_expected = 0x0606

subkeys = split_master_key(MASTER_KEY_HEX, rounds=ROUNDS)
true_final_key = subkeys[-1]

print("SBOX:       ", [f"{value:X}" for value in SBOX])
print("SBOX_INV:   ", [f"{value:X}" for value in INV_SBOX])
print("Subclaves:  ", [format_hex16(value) for value in subkeys])
print("y_r real:   ", format_hex16(true_final_key))

SBOX:        ['E', '4', 'D', '1', '2', 'F', 'B', '8', '3', 'A', '6', 'C', '5', '9', '0', '7']
SBOX_INV:    ['E', '3', '4', '8', '1', 'C', 'A', 'F', '7', 'D', '9', '6', 'B', '2', '0', '5']
Subclaves:   ['0x0011', '0x2233', '0x4455', '0x6677', '0x8899']
y_r real:    0x8899


## 2. Característica diferencial

Se fija la característica del documento:

$$\mathtt{0B00}\longrightarrow\mathtt{0040}\longrightarrow\mathtt{0220}\longrightarrow\mathtt{0606}.$$

Así, $\alpha=\alpha^{(0)}=\mathtt{0B00}$ y $\alpha^{(r-1)}=\mathtt{0606}$. Esta última diferencia activa solamente los nibbles 1 y 3. Con una sola característica se recupera la clase parcial `?8?9`, no los cuatro nibbles de `8899`.

In [89]:
characteristic = [0x0B00, 0x0040, 0x0220, 0x0606]
print(" -> ".join(f"{value:04X}" for value in characteristic))
print("Proyección activa de y_r:", f"0x{project_active_candidate(true_final_key, alpha_expected):02X}")
print("Representante canónico:   ", format_hex16(embed_active_candidate(0x89, alpha_expected)))

0B00 -> 0040 -> 0220 -> 0606
Proyección activa de y_r: 0x89
Representante canónico:    0x0809


## 3. Los dos pares de control

Para una candidata $y_r$, el descifrado parcial es

$$\widehat U=S^{-1}(C\oplus y_r).$$

El indicador $\mathrm{RP}(y_r,i)$ vale uno cuando la diferencia observada coincide con `0606` en sus nibbles activos.

In [90]:
control_plaintext_pairs = [
    (0x002B, 0x0B2B),  # par correcto
    (0x0000, 0x0B00),  # par incorrecto
]
control_pairs = encrypt_pairs(control_plaintext_pairs, subkeys)

for index, pair in enumerate(control_pairs, start=1):
    u = decrypt_last_round_partial(pair.ciphertext, true_final_key)
    u_star = decrypt_last_round_partial(pair.ciphertext_star, true_final_key)
    print(
        f"Par {index}: P={format_hex16(pair.plaintext)}, "
        f"P*={format_hex16(pair.plaintext_star)}, "
        f"C={format_hex16(pair.ciphertext)}, "
        f"C*={format_hex16(pair.ciphertext_star)}, "
        f"Delta U={format_hex16(u ^ u_star)}, "
        f"RP={right_pair(true_final_key, pair, alpha_expected)}"
    )

assert right_pair(true_final_key, control_pairs[0], alpha_expected) == 1
assert right_pair(true_final_key, control_pairs[1], alpha_expected) == 0

Par 1: P=0x002B, P*=0x0B2B, C=0x3CE1, C*=0x30ED, Delta U=0x0606, RP=1
Par 2: P=0x0000, P*=0x0B00, C=0x2F54, C*=0x2173, Delta U=0x02BB, RP=0


## 4. Generación clásica de la tabla Q1

Se usan 64 pares para mantener seis qubits de índice. Los dos pares anteriores se incluyen explícitamente y los otros 62 se generan con semilla fija. Ningún circuito consulta en superposición a $E_K$.

In [91]:
plaintext_pairs = control_plaintext_pairs + generate_plaintext_pairs(
    alpha=alpha,
    number_of_pairs=254,
    seed=2026,
)
pairs = encrypt_pairs(plaintext_pairs, subkeys)

assert len(pairs) == 256
assert all((pair.plaintext ^ pair.plaintext_star) == alpha for pair in pairs)
print(f"Pares clásicos disponibles: {len(pairs)}")

Pares clásicos disponibles: 256


## 5. Cálculo clásico de $R(y_r)$

El espacio reducido contiene las 256 combinaciones de los dos nibbles activos. Cada valor compacto `ab` se representa canónicamente como `0a0b`. El resultado recuperable correcto es la clave parcial `?8?9`; `0809` es solamente su representante interno. La característica no contiene información sobre los dos nibbles escritos como `?`.

In [92]:
K_cand = [embed_active_candidate(value, alpha_expected) for value in range(256)]
ranking = exhaustive_key_search(K_cand, pairs, alpha_expected)

for position, item in enumerate(ranking[:10], start=1):
    print(
        f"{position:2d}. candidate={format_hex16(item.candidate)}  "
        f"partial={format_partial_key(item.candidate, alpha_expected)}  R={item.R_y}"
    )

best_classical = ranking[0]
print("R(0x8899) =", count_right_pairs(true_final_key, pairs, alpha_expected))
print("Mejor representante interno:", format_hex16(best_classical.candidate))
print("Subclave realmente recuperada:", format_partial_key(best_classical.candidate, alpha_expected))

 1. candidate=0x0809  partial=0x?8?9  R=7
 2. candidate=0x0607  partial=0x?6?7  R=6
 3. candidate=0x0A04  partial=0x?A?4  R=5
 4. candidate=0x0A07  partial=0x?A?7  R=5
 5. candidate=0x0005  partial=0x?0?5  R=4
 6. candidate=0x0007  partial=0x?0?7  R=4
 7. candidate=0x0409  partial=0x?4?9  R=4
 8. candidate=0x0601  partial=0x?6?1  R=4
 9. candidate=0x0604  partial=0x?6?4  R=4
10. candidate=0x060B  partial=0x?6?B  R=4
R(0x8899) = 7
Mejor representante interno: 0x0809
Subclave realmente recuperada: 0x?8?9


## 6. Construcción de $O_2$

Para una candidata fija $x$, se calcula clásicamente la tabla $e(x,j)=\mathrm{RP}(x,j)$ y se carga como oráculo de fase:

$$O_2|j\rangle=(-1)^{e(x,j)}|j\rangle.$$

Esta es una simplificación de simulación: no se sintetiza reversiblemente el cifrador.

In [93]:
candidate = embed_active_candidate(0x89, alpha_expected)
O_2 = build_O2(candidate, pairs, alpha_expected)

print("candidate:       ", format_hex16(candidate))
print("m:               ", O_2.index_qubits)
print("N=2^m:           ", O_2.domain_size)
print("Índices marcados:", O_2.marked_indices)
print("R(candidate):    ", len(O_2.marked_indices))
print(O_2.circuit.draw(output="text"))

candidate:        0x0809
m:                8
N=2^m:            256
Índices marcados: (0, 54, 56, 141, 174, 235, 254)
R(candidate):     7
     ┌───┐          ┌───┐┌───┐               ┌───┐┌───┐               ┌───┐»
q_0: ┤ X ├───────■──┤ X ├┤ X ├────────────■──┤ X ├┤ X ├────────────■──┤ X ├»
     ├───┤       │  ├───┤└───┘            │  ├───┤└───┘            │  ├───┤»
q_1: ┤ X ├───────■──┤ X ├─────────────────■──┤ X ├─────────────────■──┤ X ├»
     ├───┤       │  ├───┤                 │  ├───┤                 │  ├───┤»
q_2: ┤ X ├───────■──┤ X ├─────────────────■──┤ X ├─────────────────■──┤ X ├»
     ├───┤       │  ├───┤┌───┐            │  ├───┤                 │  └───┘»
q_3: ┤ X ├───────■──┤ X ├┤ X ├────────────■──┤ X ├─────────────────■───────»
     ├───┤       │  ├───┤└───┘            │  └───┘                 │  ┌───┐»
q_4: ┤ X ├───────■──┤ X ├─────────────────■────────────────────────■──┤ X ├»
     ├───┤       │  ├───┤                 │                        │  ├───┤»
q_5: ┤ X ├──────

## 7. Estimación de $\widetilde R(y_r)$

La convención implementada usa $N=2^m$, $\sin^2(\theta)=M/N$ y autofases de Grover $e^{\pm 2i\theta}$. Si QPE estima $\phi=\theta/\pi$, entonces

$$\widetilde M=N\sin^2(\pi\widetilde\phi).$$

In [94]:
counting_result = quantum_counting(
    candidate,
    pairs,
    alpha_expected,
    t=6,
    debug=True,
    return_details=True,
)
counting_result

R(x)=7, R_tilde(x)=5.511637, error=1.488363, t=6, m=8, epsilon=0.015625, cota=4.715573


QuantumCountingResult(candidate=2057, real_count=7, estimated_count=5.511637026277265, absolute_error=1.4883629737227348, t=6, m=8, N=256, measured_value=3, phase=0.046875, epsilon=0.015625, count_error_bound=4.715573455530188, probabilities=(0.00779847317310471, 0.010157599038252299, 0.024691369656550527, 0.3008938937743659, 0.11757677252228643, 0.017650396288922357, 0.0070421038343983, 0.003868070288902623, 0.0024920657946579947, 0.0017659899286889794, 0.0013331063454374561, 0.0010527010842118303, 0.0008598851492980187, 0.0007212230632671116, 0.0006179932337096016, 0.0005390134129348786, 0.00047725100492472305, 0.00042809321606507056, 0.0003884072273006741, 0.00035600239351508376, 0.00032930875701507893, 0.00030717758175402164, 0.0002887535708384536, 0.00027339071086994736, 0.0002605955084285796, 0.00024998791175295223, 0.00024127394467779667, 0.00023422628439772618, 0.0002286703541371307, 0.00022447433798184106, 0.00022154206113556092, 0.00021980803226247638, 0.00021923418558948408,

## 8. Construcción de $O_1$ con una subclave umbral $y$

$O_1$ marca los índices de candidatas que cumplen $R(x)>\widetilde R(y)$. `mode="exact"` usa conteos clásicos para construir el oráculo; `mode="estimated"` ejecuta el conteo cuántico para cada candidata y debe reservarse para subconjuntos pequeños.

In [95]:
threshold = 0x0000
O_1_exact = build_O1(
    K_cand,
    threshold,
    pairs,
    alpha_expected,
    mode="exact",
)
print("y umbral:             ", format_hex16(threshold))
print("R(y):                 ", O_1_exact.threshold_score)
print("Candidatas marcadas:  ", [format_hex16(x) for x in O_1_exact.marked_candidates])

# Variante estimada, limitada a cuatro candidatas para no multiplicar el coste:
K_small = [threshold, candidate, 0x0005, 0x0607]
# O_1_estimated = build_O1(
#     K_small, threshold, pairs, alpha_expected, mode="estimated", t=6
# )

y umbral:              0x0000
R(y):                  3.0
Candidatas marcadas:   ['0x0005', '0x0007', '0x0409', '0x0601', '0x0604', '0x0607', '0x060B', '0x0704', '0x0707', '0x0803', '0x0804', '0x0806', '0x0809', '0x080E', '0x0A01', '0x0A04', '0x0A07', '0x0D09']


## 9. Búsqueda de máximo tipo Dürr--Høyer

Se parte de una candidata umbral $y$. En cada iteración se construye el conjunto de índices con conteo superior, se aplica Grover y se actualiza $y\leftarrow y'$ solamente si la medición mejora el umbral. El parámetro `c` controla el presupuesto por defecto; aquí se fija un máximo explícito.

In [96]:
maximum_result = durr_hoyer_max(
    K_cand,
    pairs,
    alpha_expected,
    mode="exact",
    initial_threshold=threshold,
    iterations=4,
    c=2.0,
    shots=2048,
    seed=2026,
)

for step in maximum_result.history:
    print(step)
print("Resultado final:", format_hex16(maximum_result.candidate), maximum_result.score)

DurrHoyerStep(iteration=0, threshold_before=0, score_before=3.0, marked_count=18, measured_index=5, measured_candidate=5, measured_score=4.0, updated=True)
DurrHoyerStep(iteration=1, threshold_before=5, score_before=4.0, marked_count=4, measured_index=103, measured_candidate=1543, measured_score=6.0, updated=True)
DurrHoyerStep(iteration=2, threshold_before=1543, score_before=6.0, marked_count=1, measured_index=137, measured_candidate=2057, measured_score=7.0, updated=True)
DurrHoyerStep(iteration=3, threshold_before=2057, score_before=7.0, marked_count=0, measured_index=None, measured_candidate=None, measured_score=None, updated=False)
Resultado final: 0x0809 7.0


## 10. Comparación final

In [97]:
correct_active = embed_active_candidate(
    project_active_candidate(true_final_key, alpha_expected),
    alpha_expected,
)

recovered_partial = format_partial_key(maximum_result.candidate, alpha_expected)
expected_partial = format_partial_key(true_final_key, alpha_expected)

print("Subclave real, solo validación:  ", format_hex16(true_final_key))
print("Resultado recuperable esperado:  ", expected_partial)
print("Representante activo esperado:   ", format_hex16(correct_active))
print("Mejor resultado clásico:         ", format_partial_key(best_classical.candidate, alpha_expected), best_classical.R_y)
print("Resultado cuántico simulado:     ", recovered_partial, maximum_result.score)
print("Coincide la subclave parcial:     ", recovered_partial == expected_partial)
print("Coincide exactamente con 0x8899: ", maximum_result.candidate == true_final_key)
print("La última comparación debe ser False: los nibbles inactivos no fueron atacados.")

Subclave real, solo validación:   0x8899
Resultado recuperable esperado:   0x?8?9
Representante activo esperado:    0x0809
Mejor resultado clásico:          0x?8?9 7
Resultado cuántico simulado:      0x?8?9 7.0
Coincide la subclave parcial:      True
Coincide exactamente con 0x8899:  False
La última comparación debe ser False: los nibbles inactivos no fueron atacados.


## 11. Por qué no se obtiene directamente `0x8899`

El indicador `RP` solo compara los nibbles activos de `0606`. En consecuencia, todas las subclaves completas con forma `a8b9` son indistinguibles para este experimento. Hay $16^2=256$ valores equivalentes y `0x8899` es uno de ellos.

Para reconstruir una clave completa hacen falta otras características que recuperen los nibbles inactivos. La celda siguiente muestra la ambigüedad y, únicamente como validación, combina el resultado `?8?9` con los nibbles inactivos conocidos del experimento. Esta recomposición no debe interpretarse como información obtenida por el ataque actual.

In [98]:
# Construir la clase desde el patrón realmente recuperado, no desde 0x?8?9.
recovered_compact = project_active_candidate(
    maximum_result.candidate,
    alpha_expected,
)
active_score = count_right_pairs(
    maximum_result.candidate,
    pairs,
    alpha_expected,
)

equivalent_full_candidates = sorted({
    embed_active_candidate(
        recovered_compact,
        alpha_expected,
        inactive_template=(high << 12) | (middle << 4),
    )
    for high in range(16)
    for middle in range(16)
})

equivalent_scores = {
    count_right_pairs(value, pairs, alpha_expected)
    for value in equivalent_full_candidates
}
same_score_for_whole_class = equivalent_scores == {active_score}
true_key_in_recovered_class = true_final_key in equivalent_full_candidates

# Solo validación controlada: los nibbles inactivos proceden de otra fuente.
inactive_template_from_other_attacks = true_final_key
reconstructed_key = embed_active_candidate(
    recovered_compact,
    alpha_expected,
    inactive_template=inactive_template_from_other_attacks,
)

print("Número de claves completas equivalentes:", len(equivalent_full_candidates))
print("Patrón recuperado por este ataque:       ", recovered_partial)
print("Conteos observados en toda la clase:     ", sorted(equivalent_scores))
print("¿Toda la clase tiene el mismo conteo?:  ", same_score_for_whole_class)
print("¿0x8899 pertenece a la clase?:          ", true_key_in_recovered_class)
print("Recomposición con información externa:  ", format_hex16(reconstructed_key))
print("¿La recomposición coincide con 0x8899?: ", reconstructed_key == true_final_key)

if not true_key_in_recovered_class:
    print(
        "Nota: la búsqueda no recuperó los nibbles activos correctos en esta "
        "ejecución; revise el ranking, la semilla o el número de pares."
    )

Número de claves completas equivalentes: 256
Patrón recuperado por este ataque:        0x?8?9
Conteos observados en toda la clase:      [7]
¿Toda la clase tiene el mismo conteo?:   True
¿0x8899 pertenece a la clase?:           True
Recomposición con información externa:   0x8899
¿La recomposición coincide con 0x8899?:  True


## 12. Limitaciones de la simulación

- Los pares y cifrados son clásicos, conforme al modelo Q1.
- Los bits de `RP` y las listas marcadas se calculan clásicamente antes de construir los oráculos.
- `Statevector` escala exponencialmente con el número de qubits.
- El espacio de subclaves se reduce a los nibbles activos de `0606`.
- Una sola característica recupera `?8?9`; otras características son necesarias para los nibbles inactivos.
- La rutina tipo Dürr--Høyer valida la estructura algorítmica, pero no demuestra una ventaja práctica ni implementa un oráculo criptográfico reversible optimizado.